In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.signal import find_peaks
from scipy import stats
pd.option_context('display.max_rows', None)
pd.option_context('display.max_columns', None)

### Load dataset, pre-process, and filter shortlisted NMIs

In [2]:
filename = Path.cwd().parent / "Dataset" / "Clariti Consumption" / "LMS_2013-01-01_2026-03-24_HALF_HOUR_au.pq"
df = pd.read_parquet(filename, engine='fastparquet')
df.columns = df.columns.str.replace(" consumption", "")

# Remove NMIs 6102507141 and VAAA003225
print("Original number of NMIs: " + str(len(df.columns)-1))
df.drop(columns=["6102507141","VAAA003225"], inplace=True)
print("New number of NMIs: "  + str(len(df.columns)-1))

df['date'] = pd.to_datetime(df['date'])
df.sort_values(by='date', ascending=True)
print(f"Duplicated period: {df[df['date'].duplicated()]['date']}")

NMI_list = df.columns.tolist()[1:]
data = {}
for nmi in NMI_list:
    first_date = min(df[df[nmi]!=0]['date'])
    last_date = max(df[df[nmi]!=0]['date'])
    filtered_col = df[((df['date']>=first_date) & (df['date']<=last_date))][[nmi]]
    
    zeroConsumption = len(filtered_col[filtered_col[nmi]==0])
    percentZeroConsumption = round(zeroConsumption*100 / len(filtered_col),2)
    
    data[nmi] = {
        'missing values':df[nmi].isna().sum(),
        'first date': first_date,
        'last date': last_date,
        '# of zero': zeroConsumption,
        '% of zero': percentZeroConsumption,
    }
    
    try:
        df.loc[df['date'] < first_date, nmi] = np.nan
    except:
        continue

summary_df = pd.DataFrame.from_dict(data, orient='index')

conditions = [(summary_df['% of zero'] < 1), (summary_df['% of zero'] < 10), (summary_df['% of zero'] < 50), (summary_df['% of zero'] < 100) ]
status = ['Active', 'Mostly Active', 'Intermittent', 'Mostly inactive']
summary_df['Status'] = np.select(conditions, status, default='Dead')
summary_df.loc[summary_df['first date']==summary_df['last date'],'Status'] = 'Dead'

NMI_shortlist = summary_df[((summary_df['Status']=='Active') | (summary_df['Status']=='Mostly Active'))].index.to_list()
filtered_df = df[NMI_shortlist + ['date']].set_index('date')

Original number of NMIs: 101
New number of NMIs: 99
Duplicated period: Series([], Name: date, dtype: datetime64[us])


In [160]:
folder = Path.cwd().parent / "Dataset"
folder.mkdir(parents=True, exist_ok=True)

### Active window

In [4]:
summary_df['Active Window'] = summary_df['first date'].astype(str) + ' to ' + summary_df['last date'].astype(str)
summary_df['Active Window']

6102000812    2013-01-01 00:00:00 to 2026-03-23 23:30:00
6102002302    2013-01-01 00:00:00 to 2026-03-23 23:30:00
6102005454    2013-01-01 00:00:00 to 2026-03-23 23:30:00
6102005592    2013-01-01 00:00:00 to 2026-03-23 23:30:00
6102009742    2013-01-01 00:00:00 to 2026-03-23 23:30:00
                                 ...                    
VAAA004066    2025-01-01 00:00:00 to 2026-03-23 23:30:00
VCCCAE0035    2013-01-01 00:00:00 to 2026-03-23 23:30:00
VCCCBC0096    2013-01-01 00:00:00 to 2026-03-23 23:30:00
VCCCSC0045    2013-01-01 00:00:00 to 2026-03-23 23:30:00
VCCCSD0058    2013-01-01 00:00:00 to 2026-03-23 23:30:00
Name: Active Window, Length: 99, dtype: str

### Zero Issues

In [5]:
# Filter zero values within the active period
data = {}
for nmi in NMI_shortlist:
    data[nmi] = df[df[nmi]==0]['date'].reset_index(drop=True)
    
zero_df = pd.DataFrame.from_dict(data)

# Calculate gaps between zeroes
# Group consecutive zeroes
data = {}
run_count = 1

for nmi in NMI_shortlist:
    if len(zero_df[nmi].dropna()) == 0:
        data[nmi + ' range'] = []
        data[nmi + ' run count'] = [0]
        continue
        
    zero_df[nmi + ' gap'] = zero_df[nmi].diff()/ pd.Timedelta(minutes=30)
    
    data[nmi + ' range'] = []
    data[nmi + ' run count'] = []
    for index, row in zero_df.iterrows():
        if index == 0:
            first_date = zero_df.loc[index,nmi]
            last_date = first_date
        elif row[nmi + ' gap'] == 1:
            last_date = zero_df.loc[index,nmi]
            run_count += 1
        else:
            data[nmi + ' range'].append(str(first_date) + ' to ' + str(last_date))
            data[nmi + ' run count'].append(run_count)
            first_date = zero_df.loc[index,nmi]
            last_date = first_date
            run_count = 1
            if row[nmi + ' gap'] != row[nmi + ' gap']:
                break
        
zero_df = pd.DataFrame.from_dict(data,orient='index').transpose()

In [6]:
# Get max and mode zero runs
# Group NMIs based on zero runs
data = {}
for nmi in NMI_shortlist:
    runs_count = zero_df[nmi + ' range'].count()
    max_run = zero_df[nmi + ' run count'].max()
    date_max_run = zero_df[zero_df[nmi + ' run count']==max_run][nmi + ' range'].tolist()

    try:
        max_run_2 = zero_df[zero_df[nmi + ' run count']!=max_run][nmi + ' run count'].max()
        date_max_run_2 = zero_df[zero_df[nmi + ' run count']==max_run_2][nmi + ' range'].tolist()
    except:
        max_run_2 = 0
        date_max_run_2 = None
        
    mode_run = zero_df[nmi + ' run count'].mode()[0]
    date_mode_run = zero_df[zero_df[nmi + ' run count']==mode_run][nmi + ' range'].tolist()
    
    data[nmi] = {
        'count of zero runs' : runs_count,
        'max zero runs' : max_run,
        'date of max zero runs' : date_max_run,
        '2nd max zero runs' : max_run_2,
        'date of 2nd max zero runs' : date_max_run_2[:10] if max_run_2 > 0 else None,
        'mode zero runs' : mode_run,
        'date of mode zero runs' : date_mode_run[:10]
    }

zero_df = pd.DataFrame.from_dict(data,orient='index')

In [7]:
zero_df.describe()

,count of zero runs,max zero runs,2nd max zero runs,mode zero runs
count,95.000000,95.000000,67.000000,95.000000
mean,135.947368,191.031579,45.014925,65.031579
std,484.674257,436.989981,31.617742,229.008754
min,0.000000,0.000000,1.000000,0.000000
25%,1.000000,88.000000,25.000000,1.000000
50%,4.000000,96.000000,48.000000,4.000000
75%,12.000000,96.000000,48.000000,48.000000
max,3357.000000,3312.000000,196.000000,1872.000000


In [8]:
data = {}
for nmi in NMI_shortlist:
    if zero_df.loc[nmi,'count of zero runs'] == 0:
        data[nmi] = 'none'
        continue
    elif zero_df.loc[nmi,'count of zero runs'] < 4:
        data[nmi] = '1-3 instances'
    elif zero_df.loc[nmi,'count of zero runs'] < 15:
        data[nmi] = '4-15 instances'
    elif zero_df.loc[nmi,'count of zero runs'] < 100:
        data[nmi] = '16-100 instances'
    else:
        data[nmi] = '100+ instances'

    if zero_df.loc[nmi,'max zero runs'] < 12:
        data[nmi] = data[nmi] + ', max <6 hours'
    elif zero_df.loc[nmi,'max zero runs'] < 48:
        data[nmi] = data[nmi] + ', max <1 day'
    elif zero_df.loc[nmi,'max zero runs'] <= 96:
        data[nmi] = data[nmi] + ', max 1-2 days'
    else:
        data[nmi] = data[nmi] + ', max 2+ days'

    if zero_df.loc[nmi,'mode zero runs'] == 1:
        data[nmi] = data[nmi] + ', mostly 30 mins'
    elif zero_df.loc[nmi,'mode zero runs'] < 12:
        data[nmi] = data[nmi] + ', mostly <6 hours'
    elif zero_df.loc[nmi,'mode zero runs'] < 48:
        data[nmi] = data[nmi] + ', mostly <1 day'
    elif zero_df.loc[nmi,'mode zero runs'] <= 96:
        data[nmi] = data[nmi] + ', mostly 1-2 days'
    else:
        data[nmi] = data[nmi] + ', mostly 2+ days'

zero_df = pd.DataFrame.from_dict(data,orient='index', columns=['Zero Issues'])
zero_df

,Zero Issues
6102000812,"4-15 instances, max 1-2 days, mostly 30 mins"
6102002302,"100+ instances, max 1-2 days, mostly 30 mins"
6102005454,"1-3 instances, max 1-2 days, mostly 30 mins"
6102005592,"4-15 instances, max 1-2 days, mostly <6 hours"
6102009742,"4-15 instances, max 1-2 days, mostly 1-2 days"
...,...
VAAA004066,none
VCCCAE0035,"16-100 instances, max 1-2 days, mostly 30 mins"
VCCCBC0096,"100+ instances, max 1-2 days, mostly 30 mins"
VCCCSC0045,"100+ instances, max 1-2 days, mostly 30 mins"


### Daily Cycle

In [9]:
dailycycle_df = filtered_df[NMI_shortlist].groupby([filtered_df.index.hour]).mean()
dailycycle_df = (dailycycle_df - dailycycle_df.mean())/ dailycycle_df.std()

In [10]:
data_peak = {}
data_valley = {}
for nmi in NMI_shortlist:
    data_peak[nmi] = dailycycle_df.index[find_peaks(dailycycle_df[nmi], prominence=0.8, distance=5)[0]]
    data_valley[nmi] = dailycycle_df.index[find_peaks(-1*dailycycle_df[nmi], prominence=0.8, distance=5)[0]]
    if len(data_peak[nmi]) == 0:
        if len(data_valley[nmi]) == 0:
            print(f'No peaks, No valleys: {nmi}')
    if len(data_peak[nmi]) == 2:
        print(f'Many peaks: {nmi} {data_peak[nmi]}')
    if len(data_valley[nmi]) == 2:
            print(f'Many valleys: {nmi} {data_valley[nmi]}')
        
peaktime_df = pd.DataFrame.from_dict(data_peak, orient='index')
valleytime_df = pd.DataFrame.from_dict(data_valley, orient='index')

Many peaks: 6103009639 Index([7, 17], dtype='int32', name='date')
Many peaks: 6103055392 Index([8, 16], dtype='int32', name='date')
Many peaks: 6103063019 Index([6, 18], dtype='int32', name='date')
Many valleys: 6103063019 Index([3, 9], dtype='int32', name='date')
Many peaks: 6203397519 Index([9, 21], dtype='int32', name='date')
Many valleys: 6203397519 Index([1, 15], dtype='int32', name='date')
Many valleys: 6203848319 Index([4, 13], dtype='int32', name='date')


In [11]:
data = {}
for nmi in NMI_shortlist:
    data[nmi] = []
   
    for time in peaktime_df.loc[nmi].dropna():
        if time < 6:
            data[nmi].append('peak in the early morning')
        elif time < 10:
            data[nmi].append('peak in the morning')
        elif time < 13:
            data[nmi].append('peak at midday')
        elif time < 16:
            data[nmi].append('peak in the afternoon')
        elif time < 19:
            data[nmi].append('peak at night')
        else:
            data[nmi].append('peak at late night')

    for time in valleytime_df.loc[nmi].dropna():
        if time < 6:
            data[nmi].append('drop in the early morning')
        elif time < 10:
            data[nmi].append('drop in the morning')
        elif time < 13:
            data[nmi].append('drop at midday')
        elif time < 16:
            data[nmi].append('drop in the afternoon')
        elif time < 19:
            data[nmi].append('drop at night')
        else:
            data[nmi].append('drop at late night')
    
    data[nmi] = list(set(data[nmi]))
    custom_order = ['peak in the early morning', 'drop in the early morning', 'peak in the morning', 'drop in the morning',
                   'peak at midday','drop at midday','peak in the afternoon', 'drop in the afternoon',
                   'peak at night','drop at night','peak at late night','drop at late night']
    data[nmi] = sorted(data[nmi], key=custom_order.index)

dailycycle_df = pd.Series(data).to_frame(name='Daily Cylce')
dailycycle_df

,Daily Cylce
6102000812,[peak in the afternoon]
6102002302,[peak in the morning]
6102005454,[peak in the afternoon]
6102005592,[peak in the afternoon]
6102009742,[peak in the afternoon]
...,...
VAAA004066,"[peak at midday, drop in the afternoon]"
VCCCAE0035,"[drop at midday, peak at night]"
VCCCBC0096,[peak at night]
VCCCSC0045,[peak in the afternoon]


### Weekday vs Weekend

In [12]:
dailycycle_dow_df = filtered_df[NMI_shortlist].groupby([filtered_df.index.dayofweek < 5, filtered_df.index.hour]).mean()
ratio_dailycycle_dow_df = dailycycle_dow_df.loc[True]/dailycycle_dow_df.loc[False]
ratio_dailycycle_dow_df.describe()

,6102000812,6102002302,6102005454,6102005592,6102009742,6102009743,6102009744,6102023971,6102038376,6102046251,...,VAAA003100,VAAA003176,VAAA003194,VAAA003197,VAAA003429,VAAA004066,VCCCAE0035,VCCCBC0096,VCCCSC0045,VCCCSD0058
count,24.000000,24.000000,24.000000,24.000000,24.000000,24.000000,24.000000,24.000000,24.000000,24.000000,...,24.000000,24.000000,24.000000,24.000000,24.000000,24.000000,24.000000,24.000000,24.000000,24.000000
mean,1.283539,2.417880,1.233785,1.637638,1.133134,1.752825,1.874165,1.186527,1.139825,1.293100,...,1.227083,2.110127,1.353479,1.356592,2.993872,1.046332,1.175131,1.125214,1.982291,1.298561
std,0.249344,1.435955,0.147877,0.409219,0.105607,0.698717,0.824679,0.140068,0.138201,0.213184,...,0.170908,0.969394,0.251821,0.155805,2.031131,0.066851,0.180935,0.106113,0.713187,0.234973
min,1.036742,1.041563,1.035372,0.986496,0.969261,0.927273,0.907035,1.029320,0.970995,0.939239,...,1.005328,1.045457,1.032040,1.093428,0.787974,1.003737,1.003161,1.009183,1.142508,1.070294
25%,1.056353,1.136716,1.103823,1.193685,1.017401,1.032535,1.037431,1.045718,1.026374,1.101868,...,1.049015,1.195942,1.110116,1.219500,1.081006,1.009348,1.019201,1.018888,1.266848,1.083614
50%,1.170887,1.745507,1.187113,1.828136,1.156602,1.912565,2.099419,1.146461,1.077965,1.364701,...,1.239915,1.851373,1.341689,1.402497,2.521419,1.012309,1.071549,1.102819,1.779611,1.223506
75%,1.566299,3.988917,1.391159,1.954006,1.223706,2.325916,2.523948,1.340144,1.291449,1.434068,...,1.412234,3.118253,1.589514,1.456840,5.189081,1.050004,1.347715,1.227268,2.747403,1.490111
max,1.658874,4.632549,1.439780,2.122262,1.277188,2.819898,3.224769,1.380592,1.344084,1.603903,...,1.429808,3.474833,1.690774,1.609358,5.769448,1.236349,1.476210,1.282601,2.950783,1.718441


In [19]:
stats = ratio_dailycycle_dow_df.describe()
data = {}
for nmi in NMI_shortlist:
    if (stats.loc['mean',nmi] <= 0.9):
        data[nmi] = 'higher weekend'
    elif (stats.loc['mean',nmi] <= 1.1):
        data[nmi] = 'similar values'
    elif stats.loc['mean',nmi] <= 1.5:
        if stats.loc['std',nmi] < 0.25:
            data[nmi] = 'slightly higher weekday, similar shape'
        elif stats.loc['std',nmi] > 1:
            data[nmi] = 'slightly higher weekday, dissimilar shape'
        else:
            data[nmi] = 'slightly higher weekday'
    else:
        data[nmi] = 'higher weekday'

dailycycle_dow_df = pd.Series(data).to_frame(name='Weekday-Weekend')
dailycycle_dow_df

,Weekday-Weekend
6102000812,"slightly higher weekday, similar shape"
6102002302,higher weekday
6102005454,"slightly higher weekday, similar shape"
6102005592,higher weekday
6102009742,"slightly higher weekday, similar shape"
...,...
VAAA004066,similar trend
VCCCAE0035,"slightly higher weekday, similar shape"
VCCCBC0096,"slightly higher weekday, similar shape"
VCCCSC0045,higher weekday


In [17]:
dailycycle_dow_df.value_counts()

Weekday-Weekend                       
slightly higher weekday, similar shape    55
higher weekday                            25
slightly higher weekday                   12
higher weekend                             3
Name: count, dtype: int64

### Seasonal Patterns

In [25]:
daily_df = filtered_df[NMI_shortlist].resample("D").sum()
rolling_mean = daily_df.rolling(window=90, min_periods=1).mean()
rolling_std = daily_df.rolling(window=30, min_periods=1).std()
seasonal_df = daily_df - rolling_mean
seasonal_df

,6102000812,6102002302,6102005454,6102005592,6102009742,6102009743,6102009744,6102023971,6102038376,6102046251,...,VAAA003100,VAAA003176,VAAA003194,VAAA003197,VAAA003429,VAAA004066,VCCCAE0035,VCCCBC0096,VCCCSC0045,VCCCSD0058
date,,,,,,,,,,,,,,,,,,,,,
2013-01-01,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
2013-01-02,78.304000,204.385500,753.479000,640.520000,7.460500,192.032000,174.331000,615.990000,219.460000,117.600000,...,1053.634500,93.596000,439.030000,1130.150000,0.000000,0.000000,165.234000,47.055000,79.616000,77.563500
2013-01-03,243.936000,313.214333,657.706667,2438.534000,-0.671000,586.284000,578.300000,1005.499333,612.519333,226.266667,...,1835.292333,280.396000,740.900000,1140.620667,0.000000,0.000000,1093.599333,174.590000,277.616000,194.580333
2013-01-04,372.430500,576.898000,445.220750,3297.249750,-4.578750,1002.721500,1002.974250,1462.741750,1311.000250,139.580000,...,2256.937750,615.798000,822.960000,568.634000,0.000000,0.000000,1790.999500,243.915000,484.854000,307.073500
2013-01-05,-130.009200,-317.906400,-496.111400,-1712.728200,-10.205400,-1050.525200,-1029.633400,-610.606600,-126.591800,-29.584000,...,-1212.720200,-691.768000,740.108800,-1384.419200,0.000000,0.000000,-428.375600,-97.032000,-709.320000,233.241200
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2026-03-19,203.014044,277.627556,843.755556,775.976667,38.945600,342.412800,315.714667,2244.246667,460.259333,742.416000,...,699.237000,399.980000,707.465333,1528.764000,25.810133,0.262578,880.881778,55.780667,231.812778,395.146667
2026-03-20,65.974756,-43.852889,681.368889,144.625556,42.397689,318.663778,276.976667,1322.818889,247.583333,365.077333,...,396.668778,77.769333,367.093778,757.682667,-28.262756,0.343200,-214.646044,17.764000,9.062000,-115.870222
2026-03-21,-92.137600,-300.115556,-74.718889,-726.562222,-0.733689,-895.070933,-872.613333,-1229.642222,-672.210667,404.111111,...,-851.273444,-605.046667,33.801333,-72.208667,-153.576889,0.207822,-542.467200,15.302000,-255.372333,-322.094933


In [132]:
from statsmodels.tsa.seasonal import seasonal_decompose
monthly_df = filtered_df[NMI_shortlist].resample("ME").sum()
data = {}
for nmi in NMI_shortlist:
    data[nmi] = seasonal_decompose(monthly_df[nmi], model='additive', period=12).seasonal[:12]

seasonality_df = pd.DataFrame.from_dict(data,orient='index').transpose().reset_index(drop=False, names=['month'])
seasonality_df['month'] = pd.to_datetime(seasonality_df['month'], format='%m').dt.month_name()
seasonality_df = seasonality_df.set_index('month')
seasonality_df = (seasonality_df - seasonality_df.min())/(seasonality_df.max() - seasonality_df.min())
seasonality_df.head(12)

,6102000812,6102002302,6102005454,6102005592,6102009742,6102009743,6102009744,6102023971,6102038376,6102046251,...,VAAA003100,VAAA003176,VAAA003194,VAAA003197,VAAA003429,VAAA004066,VCCCAE0035,VCCCBC0096,VCCCSC0045,VCCCSD0058
month,,,,,,,,,,,,,,,,,,,,,
January,0.144307,0.506832,0.125745,1.000000,0.216622,0.889116,0.932436,0.744504,0.983887,0.000000,...,0.582942,0.506456,0.194010,0.795986,0.434942,1.000000,0.975972,0.144009,0.252515,0.077741
February,0.031126,0.278859,0.003617,0.786468,0.000000,0.577055,0.614901,0.094059,0.586664,0.208556,...,0.020766,0.463055,0.233178,0.769031,0.382965,0.784602,0.358282,0.000000,0.218024,0.213601
March,0.455451,0.304147,0.978415,0.889371,0.489528,1.000000,1.000000,1.000000,1.000000,0.828425,...,0.717084,0.632627,0.910473,1.000000,0.721750,0.798337,0.716117,0.207996,0.222543,0.199090
April,0.060140,0.150561,0.554966,0.304513,0.296457,0.142015,0.174162,0.000000,0.069389,0.182984,...,0.102264,0.131381,0.593241,0.508778,0.454280,0.686915,0.224935,0.396829,0.012139,0.033627
May,0.625053,0.635452,0.967317,0.132340,0.692528,0.599113,0.552132,0.445852,0.248292,0.778286,...,0.192637,0.582621,1.000000,0.236883,0.647429,0.624636,0.845686,0.806332,0.414237,0.461010
June,0.705915,0.814620,0.325384,0.000000,0.523296,0.559846,0.505189,0.001834,0.000000,0.793295,...,0.000000,0.659609,0.708355,0.000000,0.802798,0.500012,0.717350,0.832865,0.446471,0.810261
July,1.000000,1.000000,0.495197,0.005185,0.840822,0.976544,0.882693,0.586962,0.268017,1.000000,...,0.328370,1.000000,0.750852,0.040332,1.000000,0.721313,1.000000,1.000000,1.000000,1.000000
August,0.907445,0.702892,1.000000,0.054677,1.000000,0.961803,0.871229,0.570927,0.249416,0.809047,...,0.377250,0.878760,0.924144,0.059150,0.867298,0.330110,0.740335,0.903299,0.904236,0.851930
September,0.391932,0.213799,0.652775,0.040688,0.725898,0.297913,0.250738,0.127231,0.015783,0.340831,...,0.397104,0.337500,0.637621,0.042198,0.425817,0.226770,0.000000,0.626486,0.426752,0.479085


In [133]:
data = {}
for nmi in NMI_shortlist:
    data[nmi] = seasonality_df[seasonality_df[nmi]>0.75][nmi].nlargest(2).index.tolist()
    data[nmi] = list(set(data[nmi]))
    
seasonality_df = pd.Series(data).to_frame(name='Peak Months')
seasonality_df

,Peak Months
6102000812,"[July, August]"
6102002302,"[July, June]"
6102005454,"[March, August]"
6102005592,"[January, March]"
6102009742,"[October, August]"
...,...
VAAA004066,"[January, March]"
VCCCAE0035,"[January, July]"
VCCCBC0096,"[July, August]"
VCCCSC0045,"[July, August]"


### Lag dependencies

In [144]:
# Add one exact lagged version of a variable.
def add_lag(data, variable, n):
    data[f"lag_{n}"] = data[variable].shift(n)

# Compute current-vs-past correlations for each shortlisted NMI.
lags = [1, 2, 48, 336, 1440, 17520]
rows = []

for nmi in NMI_shortlist:
    data = df[[nmi]].copy()
    for n in lags:
        add_lag(data, nmi, n)
    rows.append([nmi, *data.corr().loc[nmi, [f"lag_{n}" for n in lags]]])

corr_table = pd.DataFrame(rows, columns=["NMI", *[f"lag_{n}" for n in lags]]).round(3)
corr_table = corr_table.set_index('NMI').transpose()
corr_table

NMI,6102000812,6102002302,6102005454,6102005592,6102009742,6102009743,6102009744,6102023971,6102038376,6102046251,...,VAAA003100,VAAA003176,VAAA003194,VAAA003197,VAAA003429,VAAA004066,VCCCAE0035,VCCCBC0096,VCCCSC0045,VCCCSD0058
lag_1,0.979,0.978,0.989,0.967,0.766,0.946,0.942,0.991,0.877,0.969,...,0.998,0.949,0.963,0.961,0.969,0.784,0.955,0.966,0.978,0.936
lag_2,0.960,0.942,0.971,0.925,0.600,0.884,0.877,0.976,0.852,0.929,...,0.995,0.891,0.919,0.908,0.922,0.596,0.914,0.934,0.946,0.928
lag_48,0.799,0.691,0.828,0.744,0.627,0.755,0.750,0.813,0.732,0.736,...,0.969,0.650,0.755,0.807,0.700,0.078,0.682,0.873,0.738,0.810
lag_336,0.870,0.877,0.922,0.724,0.685,0.906,0.903,0.924,0.695,0.869,...,0.956,0.913,0.803,0.793,0.948,0.000,0.669,0.852,0.849,0.743
lag_1440,0.582,0.397,0.565,0.501,0.457,0.500,0.494,0.574,0.506,0.607,...,0.886,0.313,0.523,0.587,0.388,-0.006,0.456,0.749,0.410,0.555
lag_17520,0.641,0.594,0.618,0.557,0.419,0.608,0.596,0.711,0.350,0.561,...,0.785,0.609,0.563,0.646,0.523,-0.017,0.497,0.718,0.586,0.568


In [147]:
meaning = {
    "1": "half-hour",
    "2": "one-hour",
    "48": "day",
    "336": "week",
    "1440": "month",
    "17520": "year"
}

data = {}
for nmi in NMI_shortlist:
    high_corr = corr_table.loc[corr_table[nmi]>0.9,nmi].index.tolist()
    data[nmi] = [meaning[x.removeprefix("lag_")] for x in high_corr]

lag_df = pd.Series(data).to_frame(name='Lag Dependencies')
lag_df

,Lag Dependencies
6102000812,"[half-hour, one-hour]"
6102002302,"[half-hour, one-hour]"
6102005454,"[half-hour, one-hour, week]"
6102005592,"[half-hour, one-hour]"
6102009742,[]
...,...
VAAA004066,[]
VCCCAE0035,"[half-hour, one-hour]"
VCCCBC0096,"[half-hour, one-hour]"
VCCCSC0045,"[half-hour, one-hour]"


### Export to csv

In [161]:
old_df = pd.read_csv(Path.cwd().parent / "Dataset" / "NMI Table.csv", header=0, index_col='NMI',
                       usecols=['NMI','Mapping quality','Building Code','Building Name',
                               'Building Type','Longitude','Latitude'])

new_df = pd.merge(old_df, summary_df['Active Window'], left_index=True, right_index=True, how='outer')
new_df = pd.merge(new_df, zero_df,  left_index=True, right_index=True, how='outer')
new_df = pd.merge(new_df, dailycycle_df,  left_index=True, right_index=True, how='outer')
new_df = pd.merge(new_df, dailycycle_dow_df,  left_index=True, right_index=True, how='outer')
new_df = pd.merge(new_df, seasonality_df,  left_index=True, right_index=True, how='outer')
new_df = pd.merge(new_df, lag_df,  left_index=True, right_index=True, how='outer')
new_df.index.name = "NMI"
new_df.to_csv(folder / "NMI Summary Table.csv")
new_df

,Mapping quality,Building Code,Building Name,Building Type,Longitude,Latitude,Active Window,Zero Issues,Daily Cylce,Weekday-Weekend,Peak Months,Lag Dependencies
NMI,,,,,,,,,,,,
6102000812,unmapped,BUR,BURNLEY CAMPUS - GROUNDS,NaN,NaN,NaN,2013-01-01 00:00:00 to 2026-03-23 23:30:00,"4-15 instances, max 1-2 days, mostly 30 mins",[peak in the afternoon],"slightly higher weekday, similar shape","[July, August]","[half-hour, one-hour]"
6102002302,1:1 building,STH;298,"MELBOURNE THEATRE COMPANY HQ 252-264 STURT ST,...",Building - Minimal/Infra- Shed/Whse/Bulk Store...,144.965013,-37.828964,2013-01-01 00:00:00 to 2026-03-23 23:30:00,"100+ instances, max 1-2 days, mostly 30 mins",[peak in the morning],higher weekday,"[July, June]","[half-hour, one-hour]"
6102005454,1 building:many NMIs,PAR:110,The Spot,Building - Workspace/Teaching/Library,144.958797,-37.801606,2013-01-01 00:00:00 to 2026-03-23 23:30:00,"1-3 instances, max 1-2 days, mostly 30 mins",[peak in the afternoon],"slightly higher weekday, similar shape","[March, August]","[half-hour, one-hour, week]"
6102005592,1 building:many NMIs,PAR:110,The Spot,Building - Workspace/Teaching/Library,144.958797,-37.801606,2013-01-01 00:00:00 to 2026-03-23 23:30:00,"4-15 instances, max 1-2 days, mostly <6 hours",[peak in the afternoon],higher weekday,"[January, March]","[half-hour, one-hour]"
6102009742,1 building:many NMIs,PAR;278,"100 LEICESTER ST, CARLTON SOUTH, VIC 3053",Building - Workspace/Teaching/Library,144.960762,-37.803677,2013-01-01 00:00:00 to 2026-03-23 23:30:00,"4-15 instances, max 1-2 days, mostly 1-2 days",[peak in the afternoon],"slightly higher weekday, similar shape","[October, August]",[]
...,...,...,...,...,...,...,...,...,...,...,...,...
VAAA004066,unmapped,PAR,0,NaN,NaN,NaN,2025-01-01 00:00:00 to 2026-03-23 23:30:00,none,"[peak at midday, drop in the afternoon]",similar trend,"[January, March]",[]
VCCCAE0035,unmapped,WER,0,NaN,NaN,NaN,2013-01-01 00:00:00 to 2026-03-23 23:30:00,"16-100 instances, max 1-2 days, mostly 30 mins","[drop at midday, peak at night]","slightly higher weekday, similar shape","[January, July]","[half-hour, one-hour]"
VCCCBC0096,unmapped,CRE,0,NaN,NaN,NaN,2013-01-01 00:00:00 to 2026-03-23 23:30:00,"100+ instances, max 1-2 days, mostly 30 mins",[peak at night],"slightly higher weekday, similar shape","[July, August]","[half-hour, one-hour]"
